# Methodology

## Overview

This notebook investigates whether the sky localization regions of gravitational wave (GW) binary black hole (BBH) merger events overlap with regions of high interstellar dust column density. The motivation is that high dust density correlates with high neutral hydrogen (N_H) column density, which in turn governs the probability that X-ray photons emitted near a BBH event will scatter off interstellar dust grains before reaching Earth. Detecting such X-ray halos would allow significantly more precise localization of the gravitational wave source compared to the broad credible regions that LIGO/Virgo/KAGRA produce.

---

## Research Questions

1. **Primary**: How much do the 90% credible sky localization regions for GW BBH events overlap with sky regions that have high interstellar dust column density (as traced by neutral hydrogen N_H)?
2. **Secondary**: Among those overlapping events, what is the probability that at least one event has enough dust along the line of sight for an X-ray scattering halo to be detectable?

### Why X-ray halos from dust scattering?

When an X-ray source (such as the X-ray afterglow hypothesized to follow a BBH merger) is behind a column of interstellar dust, some X-ray photons scatter off dust grains. Because X-rays scatter at smaller angles than longer-wavelength EM radiation, scattered photons arrive at Earth along paths very slightly offset from the direct line of sight — they form a faint annular "halo" around the point source in X-ray images. The angular radius of this halo encodes the distance to the dust sheet, which can be combined with the GW distance posterior to constrain the source position to sub-arcminute precision — far better than the hundreds of square degrees typical of GW sky maps.

Two competing effects govern whether a halo is detectable:
- **Scattering optical depth (τ_scatter)**: more dust → more scattered photons → brighter halo.
- **Photoelectric absorption optical depth (τ_abs)**: more gas/dust → more X-ray photons absorbed before reaching us → fainter everything. At 1 keV, absorption dominates over scattering; at ≥2 keV, scattering wins.

---

## Libraries Used

| Library | Role in this project |
|---|---|
| `numpy` | Array operations: pixel comparisons, cumulative sums, log-space probability calculations |
| `healpy` | HEALPix sphere pixelization — reads, resamples, and reorders sky maps; converts pixel indices to (θ, φ) angles |
| `matplotlib` | All plots: equirectangular sky maps, grayscale dust overlays, scatter plots of credible regions |
| `requests` | Paginated REST API calls to GWOSC; streaming download of tar archives from Zenodo |
| `astropy.io.fits` | Reading FITS binary table files (HI4PI column density map, GW skymap inspection) |
| `astropy.coordinates.SkyCoord` | Frame conversions between equatorial (RA/Dec), galactic (l, b), and HEALPix (θ/φ) |
| `astropy.units` | Unit-safe coordinate construction (degrees, etc.) |
| `ligo.skymap.io.read_sky_map` | Reads multi-order FITS skymaps produced by LIGO/Virgo/KAGRA parameter estimation pipelines |
| `dustmaps.sfd / SFDQuery` | Queries the SFD E(B-V) optical dust reddening map — used only for qualitative background visualization (Cell 12), not for the main quantitative analysis |
| `tarfile` | Streams and extracts individual skymap `.fits.gz` files from Zenodo tar archives without loading the entire archive into RAM |
| `csv` | Writes overlap percentage results to `.csv` output files |
| `math` | `math.log` and `math.exp` for the log-space probability-of-detection calculation |

---

## Data Sources

### 1. Gravitational Wave Events — GWOSC REST API

- **URL**: `https://gwosc.org/api/v2/`
- **How fetched**: Paginated GET requests (20 events per page) to the `/catalogs/GWTC/events` endpoint with `include-default-parameters=true`. Pagination follows the `"next"` URL in each response until `"next": null`.
- **Total events fetched**: 391 (all GWTC catalogs combined: GWTC-1, GWTC-2, GWTC-2.1, GWTC-3, GWTC-4.0)
- **BBH filter**: Both component masses must exceed 3 solar masses, using `mass_1_source` and `mass_2_source` from each event's `default_parameters` list.
- **BBH events found**: 273 total; 167 have published skymaps (GWTC-1 through GWTC-4.0).

**Skymap sources by catalog:**

| Catalog | Events with skymaps | Source |
|---|---|---|
| GWTC-4.0 | 84 | Single tar.gz from Zenodo record `17602505` |
| GWTC-3-confident | 32 | Per-event URLs from GWOSC API under `"label": "skymap"` on the preferred result; tar.gz from Zenodo `8177023` |
| GWTC-2.1-confident (post-GWTC-2 events) | 41 | Per-event URLs from GWOSC API; tar.gz from Zenodo `6513631` |
| GWTC-2.1-confident (absorbed GWTC-1 events) | 10 | Zenodo record `ecf41927...` — **Note**: original GWTC-1 skymaps were on LIGO DCC but have been superseded and absorbed into GWTC-2.1 re-analyses |

### 2. Neutral Hydrogen Column Density Map — HI4PI

- **File**: `NHI_HPX.fits`
- **Survey**: HI4PI all-sky HI survey (HI4PI Collaboration 2016, A&A 594, A116)
- **Format**: HEALPix BinTable FITS, NSIDE=1024, RING ordering, galactic coordinates (l, b)
- **Column used**: `NHI` — the 21 cm integrated neutral hydrogen column density, in units of cm⁻². Each entry is indexed by `HPXINDEX` (the HEALPix pixel number at NSIDE=1024).
- **Total pixels**: 12,582,912 at NSIDE=1024
- **N_H range**: 0 to 2.398×10²² cm⁻². Negative values (unphysical noise artefacts from the radio data reduction) are clipped to 0 before use.
- **Why HI and not dust directly?** The HI4PI survey provides all-sky, uniform-resolution column density measurements. N_H traces the total gas column, and since gas and dust are well-mixed in the ISM (a standard assumption in X-ray astronomy), N_H is the standard proxy for dust column density used in X-ray scattering optical depth calculations (Draine 2003).

### 3. Optical Dust Map — SFD (visual reference only)

- **Source**: Schlegel, Finkbeiner & Davis 1998, ApJ 500, 525
- **Quantity**: E(B-V) reddening, accessed via the `dustmaps` Python package
- **Usage**: Background grayscale in Cell 12 only — qualitative visualization. **Not used in any quantitative overlap or optical depth calculation.**

---

## Key Parameters and Variables

| Variable | Value / Type | Description |
|---|---|---|
| `TARGET_NSIDE` | 1024 (integer) | HEALPix resolution for all comparisons. At NSIDE=1024, each pixel covers ≈0.003 deg² (≈11.8 arcmin²). All maps are resampled to this resolution so pixel indices are directly comparable across datasets. |
| `a` | 0.1 μm | Dust grain radius — reference value from standard ISM grain models. The scattering cross section scales as a⁴, so this choice matters. |
| `rho_grain` | 3.0 g cm⁻³ | Dust grain material density — reference value typical of silicate grains. |
| `E_keV` | 1.0 or 2.0 keV | Photon energy. 1 keV is used for the primary dust map; 2 keV is used in supplementary cells where absorption is less dominant. |
| `hi_map` | ndarray, shape (12582912,) | N_H column density per HEALPix pixel at NSIDE=1024, in cm⁻², galactic coordinates, RING ordering. |
| `tau_scattering` | ndarray, shape (12582912,) | X-ray scattering optical depth per pixel, derived from `hi_map` via the Draine (2003) formula. |
| `tau_absorption` | ndarray, shape (12582912,) | X-ray photoelectric absorption optical depth per pixel, derived from `hi_map`. |
| `scattering_mask` | bool ndarray, shape (12582912,) | `True` where scattering probability (= 1 − e^(−τ_scatter)) exceeds the chosen threshold (≥10% or ≥50%). |
| `skymap_ring` | ndarray, per event | GW source probability per pixel for one event, resampled to NSIDE=1024, RING ordering, normalized so all pixels sum to 1. |
| `confidence_map` | bool ndarray, per event | `True` for pixels inside the 90% credible region of one GW event. |
| `greatest_overlaps` | list of (str, float) tuples | `(event_name, overlap_%)` sorted descending — the main scientific result. |
| `log_prob_total_failure` | float | Sum of log(1 − overlap_i/100) across all events — used to avoid floating-point underflow when multiplying 167 small probabilities. |

---

## Coordinate Systems and Conversions

Three coordinate systems appear in this project, requiring careful and explicit conversion:

### HEALPix physics convention (healpy)
- `theta` (θ): **colatitude** — 0 at the north pole, π at the south pole
- `phi` (φ): **longitude** — 0 to 2π
- **Important**: this is NOT the same as astronomical declination. Confusing them produces silently wrong pixel lookups.

### Equatorial coordinates (GW skymaps and matplotlib plots)
- **RA** (right ascension): 0° to 360°
- **Dec** (declination): −90° to +90°
- Conversion from HEALPix angles: `dec = 90 − degrees(theta)`, `ra = degrees(phi)`

### Galactic coordinates (HI4PI)
- `l`: galactic longitude (0° to 360°)
- `b`: galactic latitude (−90° to +90°)
- Conversion via astropy `SkyCoord(ra, dec, frame='icrs').galactic.l / .b`

### Full conversion pipeline

The HI4PI map is stored in galactic HEALPix pixels; GW skymaps are in equatorial HEALPix pixels. To compare them pixel-by-pixel at the same NSIDE, the following pipeline is applied when building the visualization grid and the scattering mask:

```
Equatorial (RA, Dec)
    → astropy SkyCoord (ICRS frame)
    → galactic (l, b)
    → HEALPix physics angles (theta = 90° − b, phi = l in radians)
    → hp.ang2pix(TARGET_NSIDE, theta, phi) → pixel index in HI4PI map
```

This ensures that when we ask "does pixel i of the GW skymap overlap with pixel i of the scattering mask?", both pixel i values refer to the same patch of sky.

---

## Step-by-Step Methodology

### Step 1 — Fetch all GW events from GWOSC (Cells 2–4)

The GWOSC v2 REST API is queried with `pagesize=20` and a 0.5-second delay between pages to avoid overloading the server. Each page returns a JSON object with `"results"` (list of events) and `"next"` (URL of the next page, or `null` if done). All 391 events across GWTC-1 through GWTC-4.0 are collected.

**BBH filter** (`is_bbh` function): For each event, the `default_parameters` list is converted to a dictionary `{name: best_value}`. An event is classified as a BBH if both `mass_1_source > 3` and `mass_2_source > 3` solar masses. This threshold of 3 M☉ is the standard demarcation between neutron stars and black holes. Result: **273 BBH events**.

A second API query — targeting only catalog versions that have published skymaps (GWTC-1-confident through GWTC-4.0) — identifies the **167 BBH events** with available skymaps. For GWTC-2.1 and GWTC-3 events, individual skymap URLs are retrieved from the per-event parameters endpoint (`/api/v2/event-versions/{shortName}/parameters`), looking for the preferred result with `"label": "skymap"`. GWTC-4.0 events all share one tar archive on Zenodo.

### Step 2 — Download skymap archives (Cells 5–7)

Skymaps arrive as `.fits.gz` files inside `.tar.gz` archives on Zenodo, or as individual files linked in the GWOSC API. Archives are streamed to disk in **8 MB chunks** rather than loaded entirely into RAM, which would cause crashes for the multi-GB GWTC-4.0 archive. After streaming completes, `tarfile` is used to extract only the files whose names contain the target event name. The temp archive is deleted after extraction.

A retry mechanism (up to 3 attempts with exponential back-off: 1s, 2s, 4s) handles transient HTTP errors from the GWOSC API. Already-extracted files are skipped to make the download step idempotent (safe to re-run). All extracted files are saved in `skymaps/` as `{event_name}_skymap.fits` or `{event_name}_skymap.fits.gz`.

### Step 3 — Compute 90% credible regions (Cells 8–11)

For each skymap FITS file, the following processing pipeline is applied:

1. **Read**: `ligo.skymap.read_sky_map(file, nest=True)` returns a 1D NumPy array where each element is the posterior probability that the GW source lies within the corresponding HEALPix pixel. Pixels are in **NESTED** ordering (a space-filling-curve scheme that groups spatially adjacent pixels).

2. **Reorder to RING**: `hp.reorder(skymap, n2r=True)` converts from NESTED to **RING** ordering (the ordering used by the HI4PI map and required by most healpy visualization functions).

3. **Resample to TARGET_NSIDE=1024**: `hp.ud_grade(skymap_ring, 1024)` resamples the map — upgrading (splitting pixels) or downgrading (averaging pixels) as needed — to the common resolution. The map is renormalized (`/= skymap_ring.sum()`) after resampling because ud_grade does not guarantee exact normalization preservation.

4. **Sort descending by probability**: `np.argsort(skymap_ring)[::-1]` returns pixel indices sorted from highest to lowest probability.

5. **Cumulative sum**: `np.cumsum(skymap_ring[sorted_indices])` produces a running total of probability as pixels are added in descending probability order.

6. **Find 90% threshold**: `cumsum.searchsorted(0.90)` returns the index `threshold` at which the cumulative sum first reaches 0.90. The first `threshold` pixels in `sorted_indices` constitute the **smallest possible set of pixels whose combined probability is ≥ 90%** — this is the formal definition of a 90% credible region (also called the 90% highest-posterior-density region).

7. **Build binary confidence map**: A boolean array of the same length as the skymap is initialized to `False`. Pixels `sorted_indices[:threshold]` are set to `True`. This is `confidence_map` for that event.

### Step 4 — Build the X-ray scattering probability map (Cell 13)

This step converts the HI4PI N_H map into a sky map of X-ray scattering probability, using the analytical small-angle scattering model from Draine (2003).

#### Physical model

In the regime where the dimensionless size parameter `x = (4πa/λ) sin(θ/2) << 1` (valid for small grain sizes, high photon energies, or small scattering angles — the "Rayleigh–Gans" limit), the total scattering cross section simplifies to:

```
σ_scatter = (6.3×10⁻¹¹) · (2Z/M)² · (ρ_grain/3)² · (a/0.1μm)⁴ · (E/keV)⁻¹ · (F(E)/Z)² cm²
```

where Z is the nuclear charge of the dominant grain material, M is atomic mass, ρ_grain is grain material density, a is grain radius, E is photon energy, and F(E) is the atomic scattering factor (≈ Z far from resonances).

The dust grain number column density N_d relates to the hydrogen column density via:
```
N_d = 0.01 · N_H · m_p / m_grain    where m_grain = (4π/3) · a³ · ρ_grain
```

Combining these, the **scattering optical depth** at reference values (a=0.1μm, ρ=3 g/cm³) simplifies to:
```
τ_scatter = N_d · σ_scatter = 8.4×10⁻²³ · N_H · (a/0.1μm) · (ρ_grain/3) · (E_keV)⁻¹
```

The **photoelectric absorption optical depth** (cross section averaged over standard ISM composition) is:
```
τ_abs = 2.4×10⁻²² · N_H · (E_keV)⁻³
```

The ratio τ_scatter/τ_abs ≈ 0.35·E_keV², meaning:
- At **E = 1 keV**: absorption is ~3× stronger than scattering — X-rays are absorbed before forming a bright halo
- At **E = 2 keV**: scattering is ~1.4× stronger than absorption — halos are more detectable
- At **E ≥ 3 keV**: scattering strongly dominates, but halos become geometrically smaller and harder to resolve

The **scattering probability** for a photon traversing the full column is:
```
P_scatter = 1 − e^(−τ_scatter)
```

#### Implementation

The function `get_tau_scattering_and_absorption(a, rho_grain, E_keV)` loads `NHI_HPX.fits`, maps N_H values to their HEALPix pixel indices, clips negative values, and returns `tau_scattering` and `tau_absorption` as NumPy arrays of shape (12,582,912,) indexed in galactic HEALPix RING pixels at NSIDE=1024.

To visualize these as equirectangular images, a 1400×700 pixel grid in (RA, Dec) is constructed, converted to galactic (l, b) via astropy SkyCoord, then to HEALPix pixel indices via `hp.ang2pix`, and finally used to look up τ values. The grayscale image assigns darker tones to higher τ (higher scattering probability) using thresholds that correspond to round scattering probability values (10%, 20%, ..., 85%).

**Measured sky statistics at E=1 keV, NSIDE=1024:**
- τ_max = 2.015 (P_scatter = 86.7%), τ_mean = 0.103
- Sky fraction with τ > 1 (P_scatter > 63%): 0.6% — this is essentially the Galactic plane
- Sky fraction with τ > 0.10536 (P_scatter > 10%): 23.6%

### Step 5 — Calculate overlap percentages (Cell 15)

For each event, the fraction of the 90% credible region that falls within the high-scattering zone is computed:

```python
scattering_mask = tau_scattering > tau_threshold    # boolean, galactic HEALPix pixels
overlap_pixels  = np.sum(confidence_map & scattering_mask)
overlap_pct     = 100 * overlap_pixels / np.sum(confidence_map)
```

This pixel-wise AND operation works correctly because:
- Both arrays are at the same NSIDE (1024)
- The coordinate conversion pipeline in Step 4 ensures galactic HEALPix pixel indices in `scattering_mask` correspond to the same sky positions as equatorial HEALPix pixel indices in `confidence_map` (after conversion)

Results are sorted descending by overlap percentage and saved to:
- `bbh_overlap_percentages_>=0.5_scattering_probability.csv` (τ threshold = 0.69315, P_scatter ≥ 50%)
- `bbh_overlap_percentages_>=0.1_scattering_probability.csv` (τ threshold = 0.10536, P_scatter ≥ 10%)

**Notable result**: GW200224_222234 shows **100% overlap** at the ≥10% threshold. Its 90% credible region covers only ~50 deg² — exceptionally well-localized for a GW event — and lies entirely within the high-scattering part of the sky (near the Galactic plane).

### Step 6 — Probability of detection across all events (Cells 15.5.1–15.5.8)

Treating each event independently, the probability that **at least one** event has its 90% credible region fully (or substantially) overlapping a high-scattering zone is:

```
P(at least one success) = 1 − ∏ᵢ (1 − overlap_i / 100)
```

**Numerical issue**: direct floating-point multiplication of 167 small factors (each ≤1) underflows to exactly 0.0 in float64, making the complement 1 − 0 = 1 uninformative.

**Solution**: compute in log space using the identity log(∏ xᵢ) = Σ log(xᵢ):

```python
log_prob_total_failure = Σᵢ log(max(1 − overlap_i/100, 1e-10))
# clamp to 1e-10 to handle the 100% overlap event (log(0) = −∞)
prob_total_failure = exp(log_prob_total_failure)
prob_success = −np.expm1(log_prob_total_failure)
# np.expm1(x) = e^x − 1 with full precision near x=0,
# avoiding catastrophic cancellation when subtracting two nearly-equal floats
```

Sensitivity analysis is performed by successively halving the event list (all 167 → top 84 → top 42 → top 21) and by removing the dominant event (GW200224_222234) to assess how much one outlier drives the result. The full results table is documented in the Methodology markdown cell of the notebook.

---

## Outputs

| File | Description |
|---|---|
| `skymaps/*.fits(.gz)` | 167 individual HEALPix skymap files, one per BBH event |
| `{year}_bbh_events_90%_regions.png` | Per-year equirectangular plots of all 90% credible regions (years: 2015, 2017, 2019, 2020, 2023, 2024) |
| `scattering_probability_of_x-rays_>=_0.1.png` | Full-sky grayscale map of X-ray scattering probability at E=1 keV |
| `bbh_chunk_N_with_overlay.png` | 17 plots of 10 events each, overlaid on the dust/scattering background |
| `bbh_overlap_percentages_>=0.5_scattering_probability.csv` | Overlap percentages at ≥50% scattering threshold, sorted descending |
| `bbh_overlap_percentages_>=0.1_scattering_probability.csv` | Overlap percentages at ≥10% scattering threshold, sorted descending |
